# Test tile queue - IL sweep only

The same tiled test layout as `test_tile_queue.ipynb`, but every device is measured once on
the laser and photodiode instead of twice. Use this when the Luna OVA is not needed, or to
work on the IL sweep without spending an OVA sweep on every device first.

Writes `test_tile_queue_il.json`. The layout below - the tiles, the bottom-up reading
order, the channel derivation - is identical to the combined notebook, so a change to the
layout has to be made in both.


In [ ]:
# --- Hardware map -------------------------------------------------------------------
# Dicon GP800 switch: M ports are the instrument side, N ports the fibre-array side.
# N ports 1-7 are wired to fibre array channels 1-7.
FIBRE_ARRAY_CHANNELS = [1, 2, 3, 4, 5, 6, 7]

# Fibre array channel 4 launches light into the chip.
INPUT_CHANNEL = 4

# The IL sweep uses the laser/photodiode pair on the other two M ports: M2 carries the
# laser into the chip input, M1 returns the device output to the photodiode. M3 and M4,
# which the OVA occupies, are parked on spare channels for these measurements.
IL_LASER_M_PORT = 2
IL_PHOTODIODE_M_PORT = 1

# Measurement class name, as listed in the LabExT Experiment Wizard.
IL_MEASUREMENT_CLASS = "IL_sweep_switch"

# Instruments for the IL sweep, named here rather than left to the Experiment Wizard: the
# wizard only remembers a selection for a measurement it has been run for at least once,
# and a queue that names its own instruments loads on any machine. Values mirror
# instruments.config. The Luna sweep is left to the wizard, which already has a selection
# saved for it.
IL_INSTRUMENTS = {
    "Laser": {
        "visa": "GPIB0::23::INSTR",
        "class": "LaserMainframeKeysight",
        "channel": 0,
    },
    "Switch": {
        "visa": "None",
        "class": "SwitchDiconGP800",
        "channel": 0,
        "args": {
            "hostname": "gp800-ea6a",
            "port": 22,
            "username": "gp800admin",
            "password": "gp800admin",
        },
    },
}

# IL_sweep_switch declares six power meter roles but only opens the first "nbr of pds" of
# them. Every role still has to name an instrument or the queue will not load, so the ones
# past the four real photodiodes are pointed at the last of them and never opened.
for _role in range(6):
    _pd = min(_role, 3)
    IL_INSTRUMENTS[f"Power Meter {_role}"] = {
        "visa": "None",
        "class": "PowerMeterKoheronPD10R",
        "channel": _pd,
        "args": {"lj_port": f"AIN{_pd}"},
    }

# Must match the chip currently imported in LabExT; device ids are looked up on it.
CHIP_NAME = "test_tile_markers"

In [ ]:
# --- IL sweep settings --------------------------------------------------------------
# The band has to be one the laser can hold IL_LASER_POWER_DBM across, which is a property
# of the module's power-vs-wavelength curve rather than a free choice. The 81642A in this
# mainframe is specified 1510-1640 nm with a maximum output of +7 dBm, so 4.7 dBm over the
# C band sits inside it. The module is old though, so if a sweep is ever refused on power,
# ask the instrument what it can hold:  :sour0:wav:swe:pmax? 1530nm,1570nm
IL_LASER_POWER_DBM = 4.7                                    # dBm
IL_WAVELENGTH_START = 1530.0                                # nm, C band
IL_WAVELENGTH_STOP = 1570.0                                 # nm, C band

IL_SETTINGS = {
    "wavelength start": IL_WAVELENGTH_START,                # nm
    "wavelength stop": IL_WAVELENGTH_STOP,                  # nm
    "wavelength step": 20.0,                                # pm
    "sweep speed": 10.0,                                    # nm/s
    "sweep cycles": 1,
    "scan rate": 1000,                                      # Hz
    "laser power": IL_LASER_POWER_DBM,                      # dBm
    "nbr of pds": 1,
}


In [ ]:
# --- Queue helpers ------------------------------------------------------------------
def switch_ports(output_channel, source_m_port, receive_m_port):
    """M-port -> N-port routing that reads `output_channel` on one instrument pair.

    `source_m_port` drives the chip input channel and `receive_m_port` follows the
    device's output channel, so the same helper serves the OVA and the IL sweep - they
    differ only in which M port is the source. The remaining M ports have no instrument
    attached, but the matrix switch cannot place two M ports on the same N port, so they
    are parked on unused channels.
    """
    if output_channel not in FIBRE_ARRAY_CHANNELS:
        raise ValueError(
            f"channel {output_channel} is not a fibre array channel {FIBRE_ARRAY_CHANNELS}"
        )
    if output_channel == INPUT_CHANNEL:
        raise ValueError(
            f"channel {output_channel} is the input channel, it cannot also be an output"
        )

    ports = {source_m_port: INPUT_CHANNEL, receive_m_port: output_channel}
    spare = [c for c in FIBRE_ARRAY_CHANNELS if c not in ports.values()]
    for m_port in (1, 2, 3, 4):
        if m_port not in ports:
            ports[m_port] = spare.pop(0)
    return {f"Switch Port: M = {m}": n for m, n in sorted(ports.items())}


def move(device_id):
    """Move the stages to a device."""
    return {"type": "move", "device_id": str(device_id)}


def sfp():
    """Run a Search for Peak at the current position."""
    return {"type": "sfp"}


def il_meas(device_id, output_channel, **overrides):
    """One IL sweep of the same device, on the laser/photodiode pair instead of the OVA."""
    parameters = dict(IL_SETTINGS)
    parameters.update(switch_ports(output_channel, IL_LASER_M_PORT, IL_PHOTODIODE_M_PORT))
    parameters.update(overrides)
    return {
        "type": "meas",
        "device_id": str(device_id),
        "measurement": IL_MEASUREMENT_CLASS,
        "instruments": IL_INSTRUMENTS,
        "parameters": parameters,
    }

## The layout

One list per tile, giving the grating couplers **in the order they are drawn**, which is a
bottom-up view. `INPUT` marks the coupler the fibre array input (channel 4) is aligned to;
every other entry is the device id read out on that coupler.

Channels are derived from each device's offset from `INPUT` and mirrored, because the
drawing is bottom-up. Describe a tile in the drawn order and the routing follows.

In [4]:
INPUT = None   # marks the fibre array input coupler within a tile

# Each tile: grating couplers in drawn (bottom-up view) order, so the fibre array reads
# them in reverse. Comments give the layout label of each GC.
TILES = [
    # tile 1 - input x = 0.0 um
    #   Loopback   Input   L2chain  L8chain  L6chain
    ["4310",       INPUT,  "4520",  "4680",  "4760"],

    # tile 2 - input x = 762.0 um
    #   L4chain  Loopback  Input   L10chain  L10chain
    ["4240",     "4311",   INPUT,  "4500",   "4601"],

    # tile 3 - input x = 1397.0 um
    #   L10chain  Loopback  Input   Loopback  L4chain
    ["4202",      "4312",   INPUT,  "4514",   "4641"],

    # tile 4 - input x = 2159.0 um
    #   L6chain  L8chain  L2chain  Input   Loopback
    ["4161",     "4281",  "4321",  INPUT,  "4517"],

    # tile 5 - input x = 2667.0 um
    #   L10chain  Loopback  Input   Loopback  L4chain
    ["4203",      "4313",   INPUT,  "4515",   "4642"],

    # tile 6 - input x = 3429.0 um
    #   L6chain  L8chain  L2chain  Input   Loopback
    ["4162",     "4282",  "4322",  INPUT,  "4516"],
]

In [ ]:
# --- Build the queue ----------------------------------------------------------------
# Every device in a tile shares the tile's input coupler, so LabExT aligns once per tile
# and then measures all four devices at that one alignment. The stages are moved to the
# first device of the tile - any device of the tile would do, they share an input position.
entries = []
plan = []   # (tile number, device id, channel), kept for the preview and the cross-check

for tile_number, tile in enumerate(TILES, start=1):
    if tile.count(INPUT) != 1:
        raise ValueError(f"tile {tile_number} must mark exactly one INPUT coupler")
    input_index = tile.index(INPUT)

    tile_devices = []
    for gc_index, device_id in enumerate(tile):
        if device_id is INPUT:
            continue
        # the layout is drawn bottom-up, so array channel runs opposite to list position
        channel = INPUT_CHANNEL - (gc_index - input_index)
        if channel not in FIBRE_ARRAY_CHANNELS:
            raise ValueError(
                f"tile {tile_number}: device {device_id} maps to channel {channel}, "
                f"which is outside the fibre array {FIBRE_ARRAY_CHANNELS}"
            )
        tile_devices.append((device_id, channel))

    entries.append(move(tile_devices[0][0]))
    entries.append(sfp())
    for device_id, channel in tile_devices:
        entries.append(il_meas(device_id, channel))
        plan.append((tile_number, device_id, channel))

print(f"{len(TILES)} tiles, {len(plan)} IL sweeps, {len(entries)} queue entries")

In [ ]:
# --- Preview ------------------------------------------------------------------------
# Shows the queue the way LabExT will execute it. Measurements listed under one alignment
# step form a "block": LabExT requires every device in a block to sit at the same input
# location, since the stages are only aligned once for the whole block.
for i, entry in enumerate(entries):
    if entry["type"] == "move":
        print(f"{i:3d}  MOVE -> device {entry['device_id']}")
    elif entry["type"] == "sfp":
        print(f"{i:3d}  SEARCH FOR PEAK")
    else:
        p = entry["parameters"]
        routing = " ".join(f"M{m}=N{p[f'Switch Port: M = {m}']}" for m in (1, 2, 3, 4))
        print(f"{i:3d}      IL device {entry['device_id']:>6}  [{routing}]")

## Cross-check against the chip file (optional)

Confirms that every device named above exists on the chip and that the devices grouped
into one tile really do share an input coordinate - which is exactly what LabExT checks
when the queue is loaded. Skipped if the chip file is not next to this notebook.

In [7]:
import json
import os

CHIP_FILE = "test_tile_markers.json"

if not os.path.isfile(CHIP_FILE):
    print(f"{CHIP_FILE} not found next to this notebook - skipping cross-check.")
else:
    with open(CHIP_FILE) as f:
        chip_devices = {str(d["ID"]): d for d in json.load(f)}

    missing = sorted({device_id for _, device_id, _ in plan} - set(chip_devices))
    if missing:
        raise AssertionError(f"device ids not found in {CHIP_FILE}: {missing}")

    for tile_number, tile in enumerate(TILES, start=1):
        ids = [d for d in tile if d is not INPUT]
        positions = {tuple(chip_devices[d]["Inputs"][0]) for d in ids}
        if len(positions) != 1:
            raise AssertionError(
                f"tile {tile_number} devices do not share an input position: "
                + ", ".join(f"{d}={chip_devices[d]['Inputs'][0]}" for d in ids)
            )
        (position,) = positions
        types = ", ".join(f"{d} ({chip_devices[d]['Type']}) ch{c}"
                          for t, d, c in plan if t == tile_number)
        print(f"tile {tile_number}: input {list(position)} -> {types}")

    print("\nCross-check passed: all tiles are internally consistent.")

test_tile_markers.json not found next to this notebook - skipping cross-check.


In [ ]:
# --- Write the queue file -----------------------------------------------------------
import json

queue = {"labext_queue_version": 1, "chip_name": CHIP_NAME, "entries": entries}

out_path = "test_tile_queue_il.json"
with open(out_path, "w") as f:
    json.dump(queue, f, indent=2)

print(f"Wrote {len(entries)} entries to {out_path}")
print("Load it in LabExT via File -> Load Experiment Queue...")